# Stage 3.C — distributed aero-first BO campaign (220 mm trapezoid, COLD start)

**What this runs:** the redesigned **220 mm trapezoid** blade (ADR-0005) optimized **cold from scratch**. The earlier `blade_campaign*` / `campaign_run2` ledgers used a DIFFERENT codec (35-D vs the current 33-D) and are **void** for this geometry — this is a fresh start, NOT a continue. Two operator seeds (A = panel+ribs, B = no-rib uniform sheet) are injected ahead of a Sobol DoE via `--inject-seeds`, then BO drives toward ~300 pooled designs.

**Coordination (kept from the earlier fix):** PREP (Session 0) creates ONE fresh shared folder + a `.ready` marker; the GUARD (every session) refuses to start on duplicate folders or on a folder whose evals don't match the current codec, and locks each `SESSION_INDEX`. So the 3 sessions share ONE ledger.

**How to launch (order matters):** run **Session 0 first** through the RUN cell (its PREP creates the folder). Once it's going, run **Sessions 1 & 2** — their GUARD waits for Session 0's folder to sync, then they join. Set only `SESSION_INDEX` (0/1/2) per session.

**⚠ Coarse-tier fidelity is UNVALIDATED on this geometry:** the `N_CYCLES=3, INNER_ITER=30` coarse tier's Kendall-τ=1.0 lock was measured on the OLD 200 mm blade. Run the Stage-2 coarse-vs-fine τ check on the 220 mm geometry first if you want it confirmed before committing multi-day compute.

**Async, verified:** a seconds-long pre-flight smoke-tests the async loop before the real run; the RUN cell streams a LIVE per-completion async verdict. Resumable: re-running RUN after a Colab drop continues from the ledger (fixed total budget, so resume does not overshoot).

## 1. Repo + deps + SU2 + Drive

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")   # 1 thread/worker -> N processes on N cores

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # the Stage-3 campaign machinery is merged to main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gmsh", "cadquery"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = REPO / "data"
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO, "| drive:", DRIVE_ROOT)

In [ ]:
import urllib.request
from fanopt.cfd.phase3 import find_su2
SU2_BIN = find_su2()
if SU2_BIN is None and IN_COLAB:
    LOCAL = Path("/content/su2")
    if not any(LOCAL.rglob("SU2_CFD")):
        zc = DRIVE_ROOT / "su2" / "SU2-v8.0.1-linux64.zip"
        if not zc.exists():
            zc.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(
                "https://github.com/su2code/SU2/releases/download/v8.0.1/SU2-v8.0.1-linux64.zip", str(zc))
        LOCAL.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(zc), "-d", str(LOCAL)], check=True)
    hit = next(LOCAL.rglob("SU2_CFD"), None)
    if hit: subprocess.run(["chmod", "+x", str(hit)], check=False)
    SU2_BIN = str(hit) if hit else None
assert SU2_BIN, "SU2 not found"
print("SU2:", SU2_BIN)

## 2. Campaign config (edit `SESSION_INDEX` per session)

In [ ]:
# ---- EDIT PER SESSION -------------------------------------------------------------------
SESSION_INDEX = 0        # 0 in the FIRST session, 1 in the second, 2 in the third
N_SESSIONS    = 3        # how many Colab sessions you are running in total
SESSION_ID    = f"colab-{SESSION_INDEX}"
# ---- SHARED (identical in every session) ------------------------------------------------
N_TARGET   = 300         # TOTAL pooled designs to run (cold start). FIXED target: the run cell sets
# BUDGET = N_TARGET, so the 3 sessions collectively drive the shared ledger to N_TARGET and a resume
# after a Colab drop does NOT overshoot (it just continues toward the same total).
N_INIT     = 24          # cold-start Sobol DoE — RUNS in this cold campaign (2 injected seeds + 24 Sobol, then BO)
N_WORKERS  = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
BATCH_SIZE = N_WORKERS   # (kept for API compat) the async run (default) fills n_workers one-at-a-time,
# so real parallelism = n_workers cores; batch_size only matters to the legacy sync path.
EXPLORE_FRACTION = 0.25  # BO + exploration: 1/4 of BO-phase dispatches are space-filling Sobol draws
# (not acquisition-driven), so the search keeps probing new regions. Set 0.0 for pure exploit.
# Fidelity — coarse campaign tier. NOTE: the coarse<->fine Kendall-tau=1.0 lock was measured on the
# OLD 200 mm geometry (ADR-0004); it is NOT yet re-validated on the 220 mm trapezoid. Re-run the
# Stage-2 coarse-vs-fine tau check first if you want it confirmed. fine (5,60) is the 3.C tiebreak.
N_CYCLES   = 3           # coarse campaign tier
INNER_ITER = 30
# FRESH folder for the trapezoid COLD start — a NEW name that does NOT reuse the old-codec
# blade_campaign*/campaign_run2 ledgers (those are 35-D and void). PREP creates it empty.
SHARED_DIR = DRIVE_ROOT / "campaign_trapezoid"
print(f"session {SESSION_ID} of {N_SESSIONS} | {N_WORKERS} workers | target {N_TARGET} | shared {SHARED_DIR}")

## 3. PREP — create the fresh COLD shared folder (Session 0 only)

In [ ]:
# ============================================================================================
# PREP (Session 0 only) — creates the ONE fresh shared folder for the trapezoid COLD start and
# writes a .ready marker so Sessions 1 & 2 know it exists. NO seeding from prior runs: the old
# blade_campaign*/campaign_run2 ledgers use a DIFFERENT codec (35-D vs the current 33-D) and are
# void for this geometry. The two operator seeds (A/B) are injected at RUN time via --inject-seeds,
# then the campaign cold-starts with a Sobol DoE. Safe to Run-All in every session (non-0 no-op).
# ============================================================================================
if SESSION_INDEX == 0:
    SHARED_DIR.mkdir(parents=True, exist_ok=True)
    (SHARED_DIR / ".ready").write_text("cold-start trapezoid campaign")
    existing = list(SHARED_DIR.glob("evaluations_*.jsonl"))
    print(f"fresh COLD folder ready: {SHARED_DIR}"
          + (f"  (WARNING: {len(existing)} eval shard(s) already here — is this truly a fresh name?)"
             if existing else "  (empty — good)"))
else:
    print(f"Session {SESSION_INDEX}: PREP is Session-0 only — skipping (the guard waits for the folder).")

## 4. Safety guard — one shared folder, no duplicates (every session)

In [ ]:
# ============================================================================================
# SAFETY GUARD (run in EVERY session) — waits for the ONE shared folder, refuses to start on
# duplicate folders, verifies any existing evals match the CURRENT codec, and locks this
# SESSION_INDEX. This is the coordination fix that prevents the folder-duplication bug.
# ============================================================================================
import json, time
from fanopt.bo.blade_codec import N_DIMS
ready = SHARED_DIR / ".ready"
for _ in range(60):  # wait up to 10 min for Session 0's fresh folder to sync to this VM
    if ready.exists():
        break
    print("waiting for the shared folder from Session 0 ...", flush=True); time.sleep(10)
assert ready.exists(), "no shared folder — run PREP in Session 0 first."
dups = sorted(p for p in DRIVE_ROOT.glob(SHARED_DIR.name + "*") if p.is_dir())
assert len(dups) == 1, f"DUPLICATE shared folders exist: {dups}\nDelete the extras in Drive before launching."
# Codec guard: this is a COLD start (no seed file). A truly fresh folder has no evals yet and passes
# straight through. If it already holds evals (a resume, OR an accidentally-reused OLD folder), every
# vector MUST match the current codec or read_ledger silently drops them — the old
# blade_campaign*/campaign_run2 runs were 35-D, so pointing SHARED_DIR at one of them trips this.
bad = []
for _f in sorted(SHARED_DIR.glob("evaluations_*.jsonl")):
    for _l in _f.read_text().splitlines():
        if _l.strip():
            _v = json.loads(_l).get("vector")
            if _v is not None and len(_v) != N_DIMS:
                bad.append((_f.name, len(_v)))
            break  # one row per shard detects a codec mismatch
assert not bad, (f"existing evals are NOT {N_DIMS}-D: {bad} — this looks like an OLD (pre-trapezoid) "
    f"folder. Point SHARED_DIR at a FRESH name for the cold start.")
# lock this SESSION_INDEX so two sessions can't accidentally share an id/shard.
_lock = SHARED_DIR / f"session_{SESSION_INDEX}.lock"
try:
    with open(_lock, "x") as f: f.write(SESSION_ID)
except FileExistsError:
    if time.time() - _lock.stat().st_mtime < 300:
        raise AssertionError(f"SESSION_INDEX={SESSION_INDEX} is already in use by a session started "
                             f"<5 min ago — give THIS session a different SESSION_INDEX (0/1/2).")
    _lock.write_text(SESSION_ID)  # stale lock (a resume) — take it over
_n = sum(1 for _f in SHARED_DIR.glob('evaluations_*.jsonl')
         for _l in _f.read_text().splitlines() if _l.strip())
print(f"guard OK: one fresh folder, {_n} existing evals (all {N_DIMS}-D), SESSION_INDEX {SESSION_INDEX} locked.")

## 5. Async pre-flight — seconds, no CFD; if it fails, do not launch

In [ ]:
from fanopt.bo.distributed_campaign import preflight_async_check
# Smoke test with a FAST dummy objective (NO CFD) — confirms the async MACHINERY works in THIS Colab
# runtime BEFORE committing to the multi-hour campaign: the pool fills to N_WORKERS and REFILLS on
# completion (reaches 2*N_WORKERS unique, no duplicates). Takes ~1 min at 12 workers.
pf = preflight_async_check(n_workers=N_WORKERS)
ps = pf["per_session"]["preflight"]
print(f"async pre-flight (no CFD): peak_concurrency={ps['peak_concurrency']}/{N_WORKERS}  "
      f"refilled={pf['reached_budget']}  no_duplicates={pf['no_duplicates']}  passed={pf['passed']}")
print(f"  (smoke utilization={ps['utilization']:.0%} — LOW is EXPECTED here: the fast dummy objective "
      f"makes the serial GP proposal the bottleneck. Real async utilization is measured LIVE in the "
      f"run cell, where 2.8h evals dwarf the proposal.)")
assert pf["passed"], "ASYNC PRE-FLIGHT FAILED — pool didn't fill/refill on completion; do NOT launch."
print("OK — async dispatch-on-completion works in this runtime. Launch below; watch the run cell's "
      "live utilization verdict on the real objective.")

In [ ]:
# ---- Stage-3 trapezoid COLD rerun: inject the two operator-locked seeds ------------------
# A = panel+ribs (3 mm panel / 4 mm ribs); B = no-rib uniform 3.5 mm sheet (design B). They are
# dispatched AHEAD of the Sobol DoE via the campaign's --inject-seeds flag; both travel the normal
# claim/ledger path, so across the sessions each seed is CFD-evaluated EXACTLY ONCE and lands as an
# ordinary eval (see DistributedConfig.seed_designs / blade_campaign.stage3_seed_designs).
from fanopt.bo.blade_campaign import stage3_seed_designs
from fanopt.bo.blade_codec import N_DIMS, decode
INJECT_SEEDS = True      # set False for a pure Sobol cold-start with no seeds
_seeds = stage3_seed_designs()
for _s in _seeds:
    assert _s.shape == (N_DIMS,), f"seed is {_s.shape} but codec is {N_DIMS}-D"  # encode() => matches codec
_a, _b = decode(_seeds[0]), decode(_seeds[1])
print(f"seed injection {'ON' if INJECT_SEEDS else 'OFF'} | {len(_seeds)} seeds, each {N_DIMS}-D")
print(f"  A: ribbed  blades={_a.blade_count}  ribs={_a.t_rib_hub_m*1e3:.0f}mm  panel={_a.panel_thickness_m[0][0]*1e3:.0f}mm")
print(f"  B: uniform blades={_b.blade_count}  sheet={_b.panel_thickness_m[0][0]*1e3:.1f}mm  (no ribs)")

## 6. Run the session — streams a LIVE async verdict per completion (resumable)

In [ ]:
import time
import run_blade_campaign_distributed as campaign
import fanopt.geometry.blade_cad as blade_cad
from fanopt.bo.distributed_campaign import read_ledger
blade_cad.N_RADIAL_SECTIONS = 40  # the objective's geometry resolution (ADR-0004)

CLAIM_TTL = 6 * 3600  # must exceed the per-eval wall time — measured ~3.6-3.85h, so 6h with margin
# FIXED total budget (cold start): drive the shared ledger to N_TARGET. A resume after a Colab drop
# re-reads the ledger and continues toward the SAME total (no overshoot).
_n0 = len(read_ledger(SHARED_DIR)[0]); _t0 = time.time()
BUDGET = N_TARGET
print(f"pooled so far: {_n0} designs -> fixed total budget {BUDGET}")
argv = ["--shared-dir", str(SHARED_DIR), "--session-id", SESSION_ID,
        "--session-index", str(SESSION_INDEX), "--n-sessions", str(N_SESSIONS),
        "--budget", str(BUDGET), "--n-init", str(N_INIT), "--batch-size", str(BATCH_SIZE),
        "--n-workers", str(N_WORKERS), "--su2-bin", SU2_BIN, "--poll-seconds", "10",
        "--claim-ttl", str(CLAIM_TTL), "--explore-fraction", str(EXPLORE_FRACTION),
        # SU2 runs on LOCAL disk (fast — its live small-file I/O on the Drive mount was the earlier
        # low-utilization cause). The Drive LEDGER persists every design's parameters + result
        # (NaN for failures), which is all you need to see which failed AND 3D-render any design
        # (cells 11-12) — the disposable CFD mesh/scratch is not persisted, by design.
        "--cfd-out", "/content/cfd"]
if N_CYCLES is not None:   argv += ["--n-cycles", str(N_CYCLES)]
if INNER_ITER is not None: argv += ["--inner-iter", str(INNER_ITER)]
if globals().get("INJECT_SEEDS"):
    argv += ["--inject-seeds"]  # dispatch the two locked seeds ahead of the Sobol DoE (cell above)
# Long-running + resumable: every eval is appended to the shared Drive ledger, so re-running
# this cell after a Colab drop resumes from the ledger (no work lost).
campaign.main(argv)
_dt_h = (time.time() - _t0) / 3600; _dn = len(read_ledger(SHARED_DIR)[0]) - _n0
print(f"\nthis session: {_dn} evals in {_dt_h:.2f} h"
      + (f"  ->  ~{_dt_h / max(_dn, 1) * N_WORKERS:.2f} h/eval wall" if _dn else ""))

## 7. Campaign summary — designs, averages, progression, best (safe anytime)

In [ ]:
from fanopt.bo.campaign_analysis import campaign_report, find_shards
# The pooled current campaign (seed = all prior deduped designs + new evals). To analyze ONLY the
# first (broken) run instead: shards = [s for v in find_shards(DRIVE_ROOT,"blade_campaign*").values() for s in v]
shards = [str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")]
r = campaign_report(shards)
print(f"unique designs: {r['unique_designs']}  (finite {r['finite']}, failed/NaN {r['failed_nan']})  sources={r['sources']}")
if r.get("j_fan"):
    p = r["progression"]
    print(f"J_fan  min/mean/max : {r['j_fan']['min']:+.2e} / {r['j_fan']['mean']:+.2e} / {r['j_fan']['max']:+.2e}")
    print(f"mass g min/mean/max : {r['mass_g']['min']:.0f} / {r['mass_g']['mean']:.0f} / {r['mass_g']['max']:.0f}")
    print("\n-- did it LEARN? (means, not the lucky max) --")
    print(f"  DoE(Sobol) mean {r['sobol_mean']:+.2e} (best {r['sobol_best']:+.2e})")
    print(f"  BO         mean {r['bo_mean']:+.2e} (best {r['bo_best']:+.2e})   "
          f"-> BO proposals are {'CONSISTENTLY better' if r['bo_mean']>r['sobol_mean'] else 'NOT better'} on average")
    qm = p["quartile_means"]
    if qm: print("  mean J_fan by quarter of the run: " + " -> ".join(f"{q:+.2e}" for q in qm) +
                 "  (rising = learning)")
    print("\n-- LEARNING DEPTH (fragmentation: each session only saw its OWN history) --")
    for s, st in sorted(p["by_session"].items()):
        fh, sh = st["first_half_mean"], st["second_half_mean"]
        lift = f"{(sh-fh)/abs(fh)*100:+.0f}%" if fh else "n/a"
        print(f"  {s}: {st['n']:3} designs | 1st-half mean {fh:+.2e} -> 2nd-half {sh:+.2e} ({lift}) | best {st['best']:+.2e}")
    print(f"  => deepest single-session learning = {max(st['n'] for st in p['by_session'].values())} designs "
          f"(NOT {r['finite']}); the sessions never pooled history.")
    print(f"\nPareto front: {r['pareto_count']} designs")
    print("top by J_fan (mass is a SOFT objective — aero-first runs heavy, TO trims mass in Phase 2):")
    for d in r["top_by_j_fan"][:5]:
        print(f"  J_fan={d['j_fan']:+.2e}  mass={d['mass_g']:.0f}g  {d['source']}  {d['design_hash']}")
    print("lightest strong designs under the 300 g soft mass reference (schema.MAX_TOTAL_MASS_KG):")
    elig = r["top_under_mass_cap"]["300.0g"]
    for d in elig[:5] or [None]:
        print("  (none under 300 g — expected; the aero-first search runs heavy, TO trims mass later.)" if d is None else
              f"  J_fan={d['j_fan']:+.2e}  mass={d['mass_g']:.0f}g  {d['source']}  {d['design_hash']}")

## 8. Per-session J_fan progression (interactive — hover a dot for shape/J_fan/mass)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from fanopt.bo.campaign_analysis import session_trajectories
# INTERACTIVE per-session progression — HOVER any dot for that design's shape / J_fan / mass.
traj = session_trajectories([str(pp) for pp in SHARED_DIR.glob("evaluations_*.jsonl")])
assert traj, "no designs yet — run the campaign (cell 6) first."
palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
fig = go.Figure()
for k, (s, pts) in enumerate(sorted(traj.items())):
    col = palette[k % len(palette)]
    x = [d["eval"] for d in pts]; y = [d["j_fan"] for d in pts]
    hover = [(f"<b>{s}</b>  eval {d['eval']}<br>J_fan = {d['j_fan']:+.2e}<br>mass = {d['mass_g']:.0f} g"
              f"<br>shape = {d['peak']} / {d['interp']}<br>blades = {d['blade_count']}<br>{d['hash']}")
             for d in pts]
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name=f"{s} ({len(pts)})",
                             marker=dict(size=6, color=col, opacity=0.45),
                             text=hover, hovertemplate="%{text}<extra></extra>"))
    w = max(5, len(y) // 15)
    rm = [float(np.mean(y[max(0, i - w + 1):i + 1])) for i in range(len(y))]
    fig.add_trace(go.Scatter(x=x, y=rm, mode="lines", name=f"{s} rolling mean",
                             line=dict(color=col, width=3), hoverinfo="skip"))
fig.update_layout(title="Per-session progression — HOVER a dot for design details (shape / J_fan / mass)",
                  xaxis_title="that session's own evaluation # (time order)", yaxis_title="J_fan",
                  height=560, legend=dict(orientation="h"))
fig.show()

## 9. DESIGN progression — how the wave SHAPE evolved over the run

In [ ]:
import matplotlib.pyplot as plt
from fanopt.bo.campaign_analysis import shape_evolution
# DESIGN progression: the rib-wave SHAPE of designs sampled across the run, dark(early)->bright(late).
ev = shape_evolution([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")], n_samples=12)
assert ev, "no designs yet — run the campaign (cell 6) first."
cmap = plt.cm.viridis
plt.figure(figsize=(9, 5))
xk = range(1, len(ev[0]["knots_mm"]) + 1)
for d in ev:
    plt.plot(xk, d["knots_mm"], marker="o", color=cmap(d["eval_frac"]), alpha=0.85)
plt.xlabel("rib-bow knot  (1 = hub  ->  5 = tip)"); plt.ylabel("wave height (mm)")
plt.title("DESIGN progression — the rib-wave SHAPE over the run\n(dark = early designs -> bright = late designs)")
sm = plt.cm.ScalarMappable(cmap=cmap); sm.set_array([0, 1])
plt.colorbar(sm, ax=plt.gca(), label="run progress (early -> late)")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("Each line is one design's wave hub->tip; watch the shape shift from early(dark) to late(bright).")
for d in ev:
    print(f"  eval {d['eval_index']:3d}  peak={d['peak']:3}  J={d['j_fan']:+.2e}  knots(mm)={d['knots_mm']}")

## 10. Surface-shape coverage — RIB-bow wave + PANEL aero grid (what shapes, is it skewed?)

In [ ]:
from fanopt.bo.campaign_analysis import panel_shape_summary, shape_summary
# WHAT SHAPES did we explore, and is the pool skewed to one type? (decodes the rib-bow wave)
sh = shape_summary([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")])
assert sh["n"] > 0, "no decodable designs yet — run the campaign (cell 6) first."
print(f"designs decoded: {sh['n']}")
print(f"surface types (wave-peak/interp): {sh['type_counts']}")
print(f"wave-peak location counts       : {sh['peak_counts']}")
print("mean J_fan by peak location     : " + ", ".join(f"{k}={v:+.2e}" for k, v in sh['peak_mean_jfan'].items()))
print(f"blade_count counts              : {sh['blade_counts']}")
print(f"mean wave amplitude             : {sh['mean_amplitude_mm']:.1f} mm")
print(f"\nCONVERGENCE — did later designs drift to one type?")
print(f"  EARLY (first third) peaks: {sh['early_peak_dist']}")
print(f"  LATE  (last third)  peaks: {sh['late_peak_dist']}")
print("If LATE is dominated by one peak type, the search narrowed onto it — check whether that type "
      "actually has the best mean J_fan above, or if it just got stuck there (skew risk).")

# --- PANEL aero surface — the 24-of-33 codec dims the rib-bow summary above ignores ---------------
ps = panel_shape_summary([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")])
print(f"\nPANEL aero surface (the free 4x3 grid — camber/zigzag/multi-hump lives here):")
print(f"  panel shape types           : {ps['class_counts']}")
print(f"  panel offset amplitude       : median {ps['panel_amp_mm']['median']:.2f} mm "
      f"(p10 {ps['panel_amp_mm']['p10']:.2f}, p90 {ps['panel_amp_mm']['p90']:.2f}, max {ps['panel_amp_mm']['max']:.2f})")
print(f"  rib-bow extent (for scale)   : median {ps['rib_bow_mm_median']:.1f} mm")
print(f"  PANEL AUTHORITY vs rib bow    : {ps['authority_pct_median']:.1f}%   "
      f"(<~5% => the rib meridian dominates J_fan; the panel is a maxed-but-tiny ripple)")
print(f"  panel knobs pinned to the containment limit: {ps['at_bound_pct_median']:.0f}% of 12 nodes "
      f"(high => the optimizer WANTED more panel travel than containment allowed => the lever is "
      f"rib-thickness/containment, not the codec)")

## 11. What the TOP designs look like (the winning wave shapes)

In [ ]:
from fanopt.bo.campaign_analysis import top_designs_shapes
# What do the TOP designs actually look like? RIB-bow wave (knots hub->tip, mm) AND the PANEL shape.
for d in top_designs_shapes([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")], k=10):
    print(f"  J_fan={d['j_fan']:+.2e}  mass={d['mass_g']:3.0f}g  rib={d['peak']:3}/{d['interp']:6}  "
          f"knots(mm)={d['knots_mm']}  panel={d['panel_class']:6}({d['panel_amp_mm']:.2f}mm)  "
          f"blades={d['blade_count']}  {d['hash']}")

## 12. Which designs FAILED (from the Drive ledger)

In [ ]:
from fanopt.bo.campaign_analysis import failed_designs
# WHICH designs failed (NaN = infeasible geometry or a diverged CFD run) — read from the Drive ledger.
f = failed_designs([str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")])
print(f"{len(f)} failed designs:")
for d in f:
    print(f"  {d['hash']}  {d['source']}  peak={d['peak']}  blades={d['blade_count']}  knots(mm)={d['knots_mm']}")

## 13. Render the TOP 10 designs (3D inline + STEP export to Drive)

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import fanopt.geometry.blade_cad as blade_cad
from fanopt.geometry.blade_cad import blade_trimesh, export_blade_step
from fanopt.bo.blade_codec import decode
from fanopt.bo.campaign_analysis import classify_panel
blade_cad.N_RADIAL_SECTIONS = 40
# TOP designs by J_fan, rebuilt from ledger PARAMETERS (no CFD needed). Plotly Mesh3d => DRAG to
# rotate, SCROLL to zoom. aspectmode="data" gives TRUE proportions: the rib bow is a gentle rise on a
# 220 mm blade, NOT the exaggerated wedge matplotlib's independently-auto-scaled axes drew.
N_TOP  = 10   # how many top designs to render
N_COLS = 2    # renders PER ROW — fewer = BIGGER renders; the grid grows DOWN so you scroll through them
rows = {}
for _f in SHARED_DIR.glob("evaluations_*.jsonl"):
    for _l in _f.read_text().splitlines():
        if _l.strip():
            _d = json.loads(_l)
            if isinstance(_d.get("j_fan"), (int, float)) and _d["j_fan"] == _d["j_fan"]:
                rows[_d["design_hash"]] = _d
top = sorted(rows.values(), key=lambda d: -d["j_fan"])[:N_TOP]
assert top, "no finite designs yet — run the campaign (cell 6) first."
n_rows = -(-len(top) // N_COLS)   # ceil(len/N_COLS): enough rows to hold every design (last row may be partial)
out = SHARED_DIR / "step_exports"; out.mkdir(exist_ok=True)
meshes, titles = [], []
for k, d in enumerate(top):
    params = decode(np.array(d["vector"], dtype=float))
    v, faces = blade_trimesh(params, tol=0.001)
    meshes.append((v, faces))
    titles.append(f"#{k+1} J={d['j_fan']:+.1e}<br>{d['mass_kg']*1e3:.0f}g · panel:{classify_panel(params.panel_offsets_m)}")
    export_blade_step(params, str(out / f"top{k+1:02d}_{d['design_hash'][:8]}.step"))
# grid + placement + height ALL derive from N_COLS / n_rows, so the layout can never go out of sync.
fig = make_subplots(rows=n_rows, cols=N_COLS,
                    specs=[[{"type": "scene"}] * N_COLS for _ in range(n_rows)],
                    subplot_titles=titles, horizontal_spacing=0.02, vertical_spacing=0.03)
for k, (v, faces) in enumerate(meshes):
    fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                            intensity=v[:, 2], colorscale="Viridis", showscale=False),
                  row=k // N_COLS + 1, col=k % N_COLS + 1)
# TRUE aspect + hidden axes on every 3D scene — this is what removes the fake exaggerated angle.
fig.for_each_scene(lambda s: s.update(aspectmode="data", xaxis_visible=False,
                                      yaxis_visible=False, zaxis_visible=False))
# ~430 px PER ROW so each render is large; the figure is tall and scrolls DOWN through the rows.
fig.update_layout(height=430 * n_rows, margin=dict(l=0, r=0, t=70, b=0),
                  title_text=f"Top {len(top)} by J_fan — DRAG to rotate, SCROLL to zoom (TRUE proportions)")
fig.show()
print(f"Also wrote {len(top)} STEP files to {out} (Drive) — download for an interactive CAD viewer.")